[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/semantica-agi/semantica/blob/main/cookbook/advanced/01_Advanced_Extraction.ipynb)

> **中文译文** —— 由社区翻译，非上游原文；源文件变更后以英文原版为准。步骤间的逻辑导读为译者补充。

# 高级抽取（Advanced Extraction）

## 概览

本 notebook 演示高级语义抽取：EventDetector、CoreferenceResolver、TripletExtractor、SemanticAnalyzer、SemanticNetworkExtractor、LLMEnhancer 与 ExtractionValidator。

**文档**：[API 参考](https://semantica.readthedocs.io/reference/semantic_extract/)

### 先看主线

05/06 篇练的是抽取的基础件——圈实体、画关系。这一篇是高级件，七件工具一条流水线。三段递进：

**读懂事、认对人（第 1–2 步）—— 句子层面**

1. 事件检测：不止"谁连着谁"，还有"发生了什么"
2. 共指消解："The company" 就是 "Apple Inc."——代词和别称各归其位，图里才不会多出幽灵节点

**结构与网络（第 3–5 步）—— 从句子到全景**

3. 三元组抽取（06 篇的老朋友）
4. 语义角色分析：谁施动、谁受动
5. 语义网络：整篇文本的结构化全景

**补与验（第 6–7 步）—— 收尾**

6. LLM 增强：规则和模型抓不住的，LLM 补
7. 抽取校验：最后一道闸，分数说话

### 学习目标

- 使用 `EventDetector` 检测事件
- 使用 `CoreferenceResolver` 消解共指
- 使用 `TripletExtractor` 抽取 RDF 三元组
- 使用 `SemanticAnalyzer` 做语义分析
- 使用 `SemanticNetworkExtractor` 抽取语义网络
- 使用 `LLMEnhancer` 做 LLM 增强
- 使用 `ExtractionValidator` 校验抽取结果

## 安装

从 PyPI 安装 Semantica：

```bash
pip install semantica
# 或安装全部可选依赖：
pip install semantica[all]
```

---

## 第 1 步：事件检测（工作流：事件 → 共指 → 三元组 → 语义分析 → 网络 → LLM 增强 → 校验）

检测文本中的事件。

In [ ]:
!pip install -q semantica

In [ ]:
from semantica.semantic_extract import (
    EventDetector, CoreferenceResolver, TripletExtractor,
    SemanticAnalyzer, SemanticNetworkExtractor, LLMEnhancer, ExtractionValidator
)

text = "Apple Inc. was founded by Steve Jobs in 1976. The company is now led by Tim Cook."

event_detector = EventDetector()
events = event_detector.detect_events(text)

print(f"Detected {len(events)} events")
for event in events[:3]:
    print(f"  Event: {event.event_type} - {event.text[:50]}")


## 第 2 步：共指消解

消解文本中的共指——把"它""该公司"这类指代归位到具体实体。

In [ ]:
coreference_resolver = CoreferenceResolver()

coreferences = coreference_resolver.resolve(text)

print(f"Resolved {len(coreferences)} coreference chains")


## 第 3 步：三元组抽取

抽取 RDF 三元组——06 篇讲过的入库层标准单元，在完整流水线里的位置。

In [ ]:
triplet_extractor = TripletExtractor()

triplets = triplet_extractor.extract_triplets(text)

print(f"Extracted {len(triplets)} triplets")
for triplet in triplets[:3]:
    print(f"  ({triplet.get('subject', '')}, {triplet.get('predicate', '')}, {triplet.get('object', '')})")


## 第 4 步：语义分析

做语义角色分析——谁施动、谁受动、何时何地。

In [ ]:
semantic_analyzer = SemanticAnalyzer()

semantic_roles = semantic_analyzer.analyze_semantic_roles(text)

print(f"Analyzed semantic roles: {len(semantic_roles)}")


## 第 5 步：语义网络抽取

抽取语义网络——整篇文本的节点和边一次性成网。

In [ ]:
semantic_network_extractor = SemanticNetworkExtractor()

semantic_network = semantic_network_extractor.extract_network(text)

print(f"Extracted semantic network with {len(semantic_network.get('nodes', []))} nodes")
print(f"Edges: {len(semantic_network.get('edges', []))}")


## 第 6 步：LLM 增强

用 LLM 增强抽取结果——规则与统计模型抓不住的细节，交给 LLM 补全。

In [ ]:
llm_enhancer = LLMEnhancer()

enhanced_extractions = llm_enhancer.enhance_extractions(events, text)

print(f"Enhanced {len(enhanced_extractions)} extractions")


## 第 7 步：抽取校验

校验抽取结果——最后一道质量闸。

In [ ]:
extraction_validator = ExtractionValidator()

validation_result = extraction_validator.validate(events, text)

print(f"Extraction validation:")
print(f"  Valid: {validation_result.valid}")
print(f"  Confidence: {validation_result.confidence:.3f}")


## 小结

### 回顾主线

七件工具一条链走完：**读懂事、认对人**（事件、共指——句子层面弄清发生了什么、谁指谁）、**结构与网络**（三元组、语义角色、语义网络——从句子到全景）、**补与验**（LLM 增强、校验——补短板、把关口）。基础件加高级件，抽取侧的兵器谱齐了。

### 你已学会高级抽取能力：

- **EventDetector**：事件检测与分类
- **CoreferenceResolver**：共指消解
- **TripletExtractor**：RDF 三元组抽取
- **SemanticAnalyzer**：语义分析与角色标注
- **SemanticNetworkExtractor**：语义网络抽取
- **LLMEnhancer**：基于 LLM 的抽取增强
- **ExtractionValidator**：抽取校验